# Pelvis and Torso Bend Loss Before Landing

**Brendan Driscoll | Baseball Biomechanics Analysis | Study 11**

How much do the pelvis and torso become more upright before front-foot contact, how abruptly does that happen, and when does it occur relative to axial rotation?

This extends study 03's hinge analysis and study 05's post-plant uprighting analysis. It uses official OpenBiomechanics Dataset v1 measurements; it is not a computer-vision pelvis-angle estimator.

### Measurement scope

- The primary window is **150 ms before front-foot contact through front-foot contact** (`fp_10_time`, 10% bodyweight). It is an observation window, **not measured toe-off or the entire stride**. Sensitivities use 100 and 200 ms.
- Landing is measured separately, from front-foot contact to full foot plant (`fp_100_time`, 100% bodyweight).
- OBP pelvis X is posterior (+)/anterior (-) tilt; torso X is extension (+)/flexion (-). A positive **end minus start** change means movement toward posterior tilt/extension. We call this signed uprighting change, rather than assuming an absolute K-Vest forward-bend calibration.
- Driveline's approximately 10-degree example is a single-athlete illustration, not a diagnostic threshold. No mechanical grades or prescribed targets are assigned.
- Sustained bend-loss and rotation onsets are threshold-based analytical estimates. If motion is already above threshold at the search boundary, onset is left-censored and timing offsets are not reported.

Sources: [OBP hitting conventions](https://github.com/drivelineresearch/openbiomechanics/blob/main/baseball_hitting/README.md), [Dataset v1](https://github.com/drivelineresearch/openbiomechanics/releases/tag/dataset-v1), [Driveline article](https://drivelinebaseball.com/blogs/blog/introduction-to-hitting-biomechanics), [data license](https://github.com/drivelineresearch/openbiomechanics/blob/main/LICENSE-DATA.md).

Install the repository requirements, then run all cells. Existing inputs are reused; missing official archives download automatically. The first run needs approximately 220 MB of archives when skeleton animation is enabled. Raw inputs and individual results stay local.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib
import json
import shutil
import zipfile
import platform
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.stats import spearmanr
from scipy.signal import butter, sosfiltfilt
from IPython.display import display, HTML, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/obp_utils.py').exists()), Path.cwd())
RAW = ROOT / 'data/raw/hitting'
# Use already ignored parent directories for generated, individual-level outputs.
OUT = ROOT / 'results/hitting_phase_decomposition/stride_bend_loss'
FIG = ROOT / 'figures/hitting_phase_decomposition/stride_bend_loss'
for folder in (RAW, OUT, FIG):
    folder.mkdir(parents=True, exist_ok=True)

AUTO_DOWNLOAD = True
MAKE_SKELETON_ANIMATION = True
SELECTED_SWING = None  # Set to a processed session_swing ID, e.g. '6_1'.
LOOKBACKS_S = (0.150, 0.100, 0.200)
CUTOFF_HZ = None  # Official processed traces by default; optional 8/12/16 Hz sensitivity.
UPRIGHT_FLOOR_DPS = 20.0
ROTATION_FLOOR_DPS = 60.0
ONSET_FRACTION = 0.15
SUSTAIN_S = 0.010
BOOTSTRAPS = 1000
SEED = 20261008
EVENTS = ['fp_10_time', 'fp_100_time', 'contact_time']
ANGLE_COLS = ['pelvis_angle_x', 'torso_angle_x']
VELO_COLS = ['pelvis_angular_velocity_z', 'torso_angular_velocity_z']
OUTCOMES = ['attack_angle_contact_x', 'bat_speed_mph_contact_x',
            'blast_bat_speed_mph_x', 'exit_velo_mph_x']
ARCHIVES = {
    'joint_angles': '98a405c5c4a27c1a599d3e30b39a3087180cc4d4bb69faf62204a1c06881be95',
    'joint_velos': '3822dbce30988bc554ee533e5ccc12a5a3e92a53e185fdcb1cc8ad218dd464e4',
    'landmarks': '9cc2bdc94f5f5a6c5d540f4abe459caae2c5164cd53f6a6fd23f4b2b7283bcb3',
}
print('Primary window: 150 ms before front-foot contact; no toe-off timestamp assumed.')

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def download(url, target):
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(f'Missing {target}; provide official data or enable AUTO_DOWNLOAD.')
    print(f'Downloading {target.name} ...', flush=True)
    part = target.with_suffix(target.suffix + '.part')
    urlretrieve(url, part)
    part.replace(target)

def ensure_archive(name):
    target = RAW / (name + '.csv')
    if target.exists():
        return target
    archive = RAW / ('hitting_' + name + '.zip')
    if not archive.exists():
        download('https://github.com/drivelineresearch/openbiomechanics/releases/download/'
                 'dataset-v1/' + archive.name, archive)
    if sha256(archive) != ARCHIVES[name]:
        raise ValueError(f'{archive.name}: official archive checksum mismatch.')
    with zipfile.ZipFile(archive) as z:
        matches = [n for n in z.namelist() if Path(n).name == target.name]
        if len(matches) != 1:
            raise ValueError(f'Expected exactly one {target.name}.')
        part = target.with_suffix('.csv.part')
        with z.open(matches[0]) as source, part.open('wb') as dest:
            shutil.copyfileobj(source, dest)
        part.replace(target)
    return target

for name in ('joint_angles', 'joint_velos'):
    ensure_archive(name)
csv_base = ('https://raw.githubusercontent.com/drivelineresearch/openbiomechanics/'
            'dataset-v1/baseball_hitting/data/')
for name, remote in [('metadata.csv', 'metadata.csv'), ('poi_metrics.csv', 'poi/poi_metrics.csv')]:
    if not (RAW / name).exists():
        download(csv_base + remote, RAW / name)

metadata = pd.read_csv(RAW / 'metadata.csv', dtype={'session_swing': str, 'user': str, 'session': str})
poi = pd.read_csv(RAW / 'poi_metrics.csv', dtype={'session_swing': str})
angles = pd.read_csv(RAW / 'joint_angles.csv', usecols=['session_swing', 'time'] + EVENTS + ANGLE_COLS,
                     dtype={'session_swing': str})
velos = pd.read_csv(RAW / 'joint_velos.csv', usecols=['session_swing', 'time'] + EVENTS + VELO_COLS,
                    dtype={'session_swing': str})
for name, table, keys in [('metadata', metadata, ['session_swing']), ('POI', poi, ['session_swing']),
                          ('angles', angles, ['session_swing', 'time']),
                          ('velocities', velos, ['session_swing', 'time'])]:
    if table[keys].isna().any().any() or table.duplicated(keys).any():
        raise ValueError(f'{name}: missing or duplicate join keys.')
swings = metadata[['session_swing', 'user', 'session', 'hitter_side']].merge(
    poi[['session_swing'] + OUTCOMES], on='session_swing', how='left', validate='one_to_one')
angle_groups = {k: g.sort_values('time') for k, g in angles.groupby('session_swing', sort=False)}
velo_groups = {k: g.sort_values('time') for k, g in velos.groupby('session_swing', sort=False)}
print(f'Inputs: {len(swings):,} swings from {swings.user.nunique():,} hitters.')

## Extraction and timing rules

Signals are evaluated on their own time grids; event boundaries are interpolated exactly. Angles are unwrapped before differencing. No missing values or sampling gaps are bridged. The default uses the official processed signals without an additional filter. Optional filtering is offline, zero-phase, and may shift estimated onset.

For each segment, net uprighting is the signed angle difference; maximum excursion is the greatest increase above the window-start angle. Positive-rate duration sums intervals with increasing angle. Peak rate is computed from the time derivative of the angle trace, not from the axial-rotation velocity channel.

Onset requires a sustained run above the larger of an absolute floor and 15% of the positive peak within the search window (window start to ball contact). Search boundaries and thresholds are analytical choices. A run spanning the left boundary is marked censored. A missing onset means no qualifying run, not proof that movement did not occur.

In [ ]:
def validate_trace(group, columns):
    t = group.time.to_numpy(float)
    if len(t) < 4 or not np.isfinite(t).all() or np.any(np.diff(t) <= 0):
        raise ValueError('invalid time grid')
    dt = np.median(np.diff(t))
    if not np.allclose(np.diff(t), dt, rtol=0.05, atol=1e-6):
        raise ValueError('irregular sampling or gap')
    if not np.isfinite(group[columns].to_numpy(float)).all():
        raise ValueError('missing/nonfinite signal')
    return t

def get_events(group):
    values = group[EVENTS].to_numpy(float)
    if not np.isfinite(values).all():
        raise ValueError('missing event')
    if not np.allclose(values, values[0], rtol=0, atol=1e-6):
        raise ValueError('inconsistent events within swing')
    ffc, plant, contact = values[0]
    if not ffc < plant < contact:
        raise ValueError('unordered or zero-duration events')
    return values[0]

def boundary_window(t, values, start, end):
    if start < t[0] - 1e-9 or end > t[-1] + 1e-9 or end <= start:
        raise ValueError('window outside sampled range or unordered')
    if end - start < 2 * np.median(np.diff(t)):
        raise ValueError('window shorter than two sampling intervals')
    grid = np.r_[start, t[(t > start) & (t < end)], end]
    return grid, np.interp(grid, t, values)

def angle_trace(t, raw, cutoff=None):
    x = np.rad2deg(np.unwrap(np.deg2rad(raw)))
    if cutoff is not None:
        fs = 1 / np.median(np.diff(t))
        if not 0 < cutoff < fs / 2:
            raise ValueError('invalid filter cutoff')
        x = sosfiltfilt(butter(4, cutoff, fs=fs, output='sos'), x)
    return x, np.gradient(x, t)

def sustained_onset(t, rate, start, end, floor, fraction=ONSET_FRACTION, sustain=SUSTAIN_S):
    grid, y = boundary_window(t, rate, start, end)
    threshold = max(floor, fraction * max(0.0, float(y.max())))
    above = y >= threshold
    starts = np.flatnonzero(above & ~np.r_[False, above[:-1]])
    for i in starts:
        j = i
        while j + 1 < len(grid) and above[j + 1]:
            j += 1
        if grid[j] - grid[i] >= sustain - 1e-9:
            if i == 0:
                return np.nan, 'left_censored', threshold
            return float(grid[i]), 'detected', threshold
    return np.nan, 'not_detected', threshold

def positive_duration(grid, values):
    return float(np.diff(grid)[np.diff(values) > 1e-10].sum())

def extract_swing(a, v, lookback=.150, cutoff=None,
                  upright_floor=UPRIGHT_FLOOR_DPS, rotation_floor=ROTATION_FLOOR_DPS):
    ea, ev = get_events(a), get_events(v)
    if not np.allclose(ea, ev, rtol=0, atol=1e-6):
        raise ValueError('events disagree between angle and velocity tables')
    ffc, plant, contact = ea
    start = ffc - lookback
    # Validate requested windows, not unused velocity endpoints (which can be NaN).
    def crop(g, left, right):
        t = g.time.to_numpy(float)
        if not np.isfinite(t).all() or np.any(np.diff(t) <= 0):
            raise ValueError('invalid time grid')
        if left < t[0] or right > t[-1]:
            raise ValueError('window outside sampled range')
        lo = max(0, np.searchsorted(t, left, side='right') - 1)
        hi = min(len(t), np.searchsorted(t, right, side='left') + 1)
        return g.iloc[lo:hi]
    pad = .100 if cutoff is not None else 0.0
    a = crop(a, start - pad, contact + pad)
    v = crop(v, start, contact)
    ta = validate_trace(a, ANGLE_COLS)
    tv = validate_trace(v, VELO_COLS)
    # Require complete signal coverage through contact in both tables.
    boundary_window(ta, a.pelvis_angle_x.to_numpy(float), start, contact)
    boundary_window(tv, v.pelvis_angular_velocity_z.to_numpy(float), start, contact)
    result = dict(start_time=start, ffc_time=ffc, plant_time=plant, contact_time=contact,
                  lookback_ms=1000 * lookback, landing_duration_ms=1000 * (plant - ffc))
    traces = dict(angle_time=ta, velo_time=tv)
    for segment in ('pelvis', 'torso'):
        x, rate = angle_trace(ta, a[f'{segment}_angle_x'].to_numpy(float), cutoff)
        rotation = v[f'{segment}_angular_velocity_z'].to_numpy(float)
        grid, pre = boundary_window(ta, x, start, ffc)
        _, pre_rate = boundary_window(ta, rate, start, ffc)
        land_t, land = boundary_window(ta, x, ffc, plant)
        peak_i = int(np.argmax(pre_rate))
        prefix = segment + '_'
        result.update({prefix + 'angle_start_deg': pre[0], prefix + 'angle_ffc_deg': pre[-1],
                       prefix + 'angle_plant_deg': land[-1],
                       prefix + 'pre_uprighting_deg': pre[-1] - pre[0],
                       prefix + 'pre_max_excursion_deg': max(0.0, float(pre.max() - pre[0])),
                       prefix + 'landing_uprighting_deg': land[-1] - land[0],
                       prefix + 'start_to_plant_uprighting_deg': land[-1] - pre[0],
                       prefix + 'pre_peak_rate_dps': pre_rate[peak_i],
                       prefix + 'pre_peak_rate_ms_from_ffc': 1000 * (grid[peak_i] - ffc),
                       prefix + 'pre_positive_rate_duration_ms': 1000 * positive_duration(grid, pre)})
        bend_on, bend_status, bend_thr = sustained_onset(ta, rate, start, contact, upright_floor)
        rot_on, rot_status, rot_thr = sustained_onset(tv, rotation, start, contact, rotation_floor)
        offset = 1000 * (bend_on - rot_on) if bend_status == rot_status == 'detected' else np.nan
        result.update({prefix + 'bend_onset_time': bend_on, prefix + 'bend_onset_status': bend_status,
                       prefix + 'rotation_onset_time': rot_on, prefix + 'rotation_onset_status': rot_status,
                       prefix + 'bend_threshold_dps': bend_thr, prefix + 'rotation_threshold_dps': rot_thr,
                       prefix + 'bend_minus_rotation_ms': offset,
                       prefix + 'bend_onset_ms_from_ffc': 1000 * (bend_on - ffc),
                       prefix + 'rotation_onset_ms_from_ffc': 1000 * (rot_on - ffc)})
        if bend_status == 'detected' and rot_status == 'detected' and rot_on > start:
            end = min(rot_on, ffc)
            end_angle = np.interp(end, ta, x)
            result[prefix + 'net_change_before_rotation_deg'] = end_angle - pre[0]
        else:
            result[prefix + 'net_change_before_rotation_deg'] = np.nan
        traces[prefix + 'angle'] = x
        traces[prefix + 'rate'] = rate
        traces[prefix + 'rotation'] = rotation
    if not np.isfinite([result[s + '_pre_uprighting_deg'] for s in ('pelvis', 'torso')]).all():
        raise ValueError('nonfinite extracted features')
    return result, traces

def extract_cohort(lookback, cutoff=None, upright_floor=UPRIGHT_FLOOR_DPS,
                   rotation_floor=ROTATION_FLOOR_DPS):
    rows, audit = [], []
    for meta in swings.to_dict('records'):
        sid = meta['session_swing']
        reason = 'retained'
        try:
            if pd.isna(meta['user']) or pd.isna(meta['session']):
                raise ValueError('missing athlete/session identity')
            if sid not in angle_groups or sid not in velo_groups:
                raise ValueError('missing signal table')
            features, _ = extract_swing(angle_groups[sid], velo_groups[sid], lookback,
                                        cutoff, upright_floor, rotation_floor)
            rows.append({**meta, **features})
        except ValueError as e:
            reason = str(e)
        audit.append(dict(session_swing=sid, user=meta['user'], lookback_ms=lookback * 1000,
                          cutoff_hz=cutoff, reason=reason))
    return pd.DataFrame(rows), pd.DataFrame(audit)

results, audit = extract_cohort(LOOKBACKS_S[0], CUTOFF_HZ)
if results.empty:
    raise ValueError('No retained swings. Inspect event order, coverage, and local input files.')
display(audit.reason.value_counts().rename_axis('reason').to_frame('swings'))
print(f'Retained {len(results):,} swings from {results.user.nunique():,} hitters.')
display(results[['session_swing', 'user', 'pelvis_pre_uprighting_deg',
                 'torso_pre_uprighting_deg', 'pelvis_bend_minus_rotation_ms']].head().round(2))

## Hitter profiles and selected-swing timeline

Hitter/session means and standard deviations describe repeatability within this assessment. They do not estimate reliability across days. The default example is the swing closest to median pelvis pre-contact change, selected without using outcomes.

In [ ]:
FEATURES = ['pelvis_pre_uprighting_deg', 'torso_pre_uprighting_deg',
            'pelvis_pre_max_excursion_deg', 'torso_pre_max_excursion_deg',
            'pelvis_landing_uprighting_deg', 'torso_landing_uprighting_deg',
            'pelvis_pre_peak_rate_dps', 'torso_pre_peak_rate_dps',
            'pelvis_pre_positive_rate_duration_ms', 'torso_pre_positive_rate_duration_ms',
            'pelvis_bend_minus_rotation_ms', 'torso_bend_minus_rotation_ms']
profiles = results.groupby(['user', 'session'])[FEATURES + OUTCOMES].agg(['mean', 'std', 'count'])
profiles.columns = ['_'.join(c) for c in profiles.columns]
profiles = profiles.reset_index()
display(results[FEATURES].describe(percentiles=[.1, .5, .9]).T.round(2))
status_counts = pd.concat([
    results[f'{s}_{kind}_onset_status'].value_counts().rename(f'{s}_{kind}')
    for s in ('pelvis', 'torso') for kind in ('bend', 'rotation')], axis=1).fillna(0).astype(int)
display(status_counts)

if SELECTED_SWING is None:
    selected_id = results.loc[(results.pelvis_pre_uprighting_deg -
                               results.pelvis_pre_uprighting_deg.median()).abs().idxmin(), 'session_swing']
else:
    selected_id = str(SELECTED_SWING)
if selected_id not in set(results.session_swing):
    raise ValueError(f'Selected swing {selected_id} is not retained; see exclusion audit.')
selected = results.set_index('session_swing').loc[selected_id]
_, trace = extract_swing(angle_groups[selected_id], velo_groups[selected_id], LOOKBACKS_S[0], CUTOFF_HZ)
colors = {'pelvis': '#1565C0', 'torso': '#A1263B'}

def mark_events(ax):
    ax.axvline(0, color='#222222', ls='--', lw=1, label='Front-foot contact')
    ax.axvline(1000 * (selected.plant_time - selected.ffc_time), color='#777777', ls=':',
               lw=1, label='Full foot plant')
    ax.axvline(1000 * (selected.contact_time - selected.ffc_time), color='#999999', ls='-.',
               lw=1, label='Ball contact')
    ax.axvspan(-selected.lookback_ms, 0, color='#dddddd', alpha=.22)

fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)
for segment in ('pelvis', 'torso'):
    ta = 1000 * (trace['angle_time'] - selected.ffc_time)
    tv = 1000 * (trace['velo_time'] - selected.ffc_time)
    axes[0].plot(ta, trace[segment + '_angle'] - selected[segment + '_angle_start_deg'],
                 color=colors[segment], label=segment.title())
    axes[1].plot(ta, trace[segment + '_rate'], color=colors[segment], label=segment.title())
    axes[2].plot(tv, trace[segment + '_rotation'], color=colors[segment], label=segment.title())
    for kind, ax in [('bend', axes[1]), ('rotation', axes[2])]:
        if selected[segment + '_' + kind + '_onset_status'] == 'detected':
            when = 1000 * (selected[segment + '_' + kind + '_onset_time'] - selected.ffc_time)
            ax.axvline(when, color=colors[segment], alpha=.55, ls='--')
for ax, ylabel in zip(axes, ['Change from window start (deg)', 'Uprighting rate (deg/s)',
                            'Axial rotation velocity (deg/s)']):
    mark_events(ax)
    ax.axhline(0, color='#cccccc', lw=.8)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=.15)
axes[0].legend(ncol=3, fontsize=8)
axes[2].set_xlim(-selected.lookback_ms, 1000 * (selected.contact_time - selected.ffc_time))
axes[2].set_xlabel('Time from front-foot contact (ms)')
fig.suptitle(f'Swing {selected_id}: posture change before landing and rotation')
fig.tight_layout()
fig.savefig(FIG / 'selected_swing_timeline.png', dpi=160)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for segment in ('pelvis', 'torso'):
    axes[0].hist(results[segment + '_pre_uprighting_deg'], bins=25, alpha=.45,
                 color=colors[segment], label=segment.title())
    valid = results[[segment + '_pre_uprighting_deg', segment + '_pre_peak_rate_dps']].dropna()
    axes[1].scatter(valid.iloc[:, 0], valid.iloc[:, 1], s=14, alpha=.35,
                    color=colors[segment], label=segment.title())
axes[0].set(xlabel='Signed pre-contact uprighting change (deg)', ylabel='Swings')
axes[1].set(xlabel='Signed pre-contact uprighting change (deg)', ylabel='Peak rate (deg/s)')
for ax in axes:
    ax.legend()
    ax.grid(alpha=.15)
fig.tight_layout()
fig.savefig(FIG / 'cohort_posture_changes.png', dpi=160)
plt.show()

## Optional synchronized skeleton animation

This is a reconstruction from measured joint centers, not synchronized camera video. The display projects onto global Y–Z (across the batter's box and upward), approximately the hitter's sagittal view before rotation. Projection changes as the hitter rotates; it is **not** a body-local pelvis tilt estimate. The hip-to-shoulder line visualizes trunk geometry only. Numerical pelvis/torso changes come from their independent OBP angle traces. No pelvis-angle ray is invented from two hip centers.

The animation runs from window start through ball contact and shares a cursor with the angle-change plot. Set `MAKE_SKELETON_ANIMATION=False` to skip the extra archive.

In [ ]:
animation_html = None
if MAKE_SKELETON_ANIMATION:
    landmark_path = ensure_archive('landmarks')
    bases = ['left_hip', 'right_hip', 'lsjc', 'rsjc', 'lkjc', 'rkjc', 'lajc', 'rajc']
    cols = [f'{base}_{axis}' for base in bases for axis in 'xyz']
    chunks = []
    for chunk in pd.read_csv(landmark_path, usecols=['session_swing', 'time'] + cols,
                             dtype={'session_swing': str}, chunksize=100000):
        part = chunk.loc[chunk.session_swing.eq(selected_id)]
        if len(part):
            chunks.append(part)
    if not chunks:
        print('Selected swing absent from landmarks; angle plots remain available.')
    else:
        lm = pd.concat(chunks).sort_values('time')
        if lm.duplicated(['session_swing', 'time']).any():
            raise ValueError('Duplicate landmark samples.')
        lt = validate_trace(lm, cols)
        if lt[0] > selected.start_time or lt[-1] < selected.contact_time:
            raise ValueError('Landmark coverage does not match selected animation window.')
        # Uniform 60 fps presentation; angle and landmark signals interpolate independently.
        frame_times = np.linspace(selected.start_time, selected.contact_time,
                                  max(2, int(np.ceil((selected.contact_time - selected.start_time) * 60)) + 1))
        points = {b: np.column_stack([np.interp(frame_times, lt, lm[f'{b}_{a}']) for a in 'yz'])
                  for b in bases}
        links = [('left_hip', 'right_hip'), ('lsjc', 'rsjc'), ('left_hip', 'lsjc'),
                 ('right_hip', 'rsjc'), ('left_hip', 'lkjc'), ('lkjc', 'lajc'),
                 ('right_hip', 'rkjc'), ('rkjc', 'rajc')]
        fig, (body_ax, angle_ax) = plt.subplots(1, 2, figsize=(12, 5))
        all_points = np.vstack(list(points.values()))
        low, high = all_points.min(axis=0), all_points.max(axis=0)
        margin = np.maximum((high - low) * .15, .08)
        body_ax.set(xlim=(low[0] - margin[0], high[0] + margin[0]),
                    ylim=(low[1] - margin[1], high[1] + margin[1]),
                    xlabel='Global Y (m)', ylabel='Upward Z (m)')
        body_ax.set_aspect('equal')
        lines = [body_ax.plot([], [], color='#444444', lw=2.5)[0] for _ in links]
        trunk_line, = body_ax.plot([], [], color='#A1263B', lw=3, label='Hip-to-shoulder line')
        body_ax.legend(fontsize=8)
        title = body_ax.set_title('Skeleton view', fontsize=10)
        for segment in ('pelvis', 'torso'):
            angle_ax.plot(1000 * (trace['angle_time'] - selected.ffc_time),
                          trace[segment + '_angle'] - selected[segment + '_angle_start_deg'],
                          color=colors[segment], label=segment.title())
        mark_events(angle_ax)
        angle_ax.set(xlim=(-selected.lookback_ms, 1000 * (selected.contact_time - selected.ffc_time)),
                     xlabel='Time from front-foot contact (ms)', ylabel='Signed angle change (deg)')
        angle_ax.legend(fontsize=8)
        cursor = angle_ax.axvline(-selected.lookback_ms, color='black', lw=1.5)
        angle_text = angle_ax.text(.02, .97, '', transform=angle_ax.transAxes, va='top')
        def update_frame(i):
            for line, (a, b) in zip(lines, links):
                xy = np.vstack([points[a][i], points[b][i]])
                line.set_data(xy[:, 0], xy[:, 1])
            hip = (points['left_hip'][i] + points['right_hip'][i]) / 2
            shoulder = (points['lsjc'][i] + points['rsjc'][i]) / 2
            trunk_line.set_data([hip[0], shoulder[0]], [hip[1], shoulder[1]])
            now = frame_times[i]
            ms = 1000 * (now - selected.ffc_time)
            cursor.set_xdata([ms, ms])
            changes = [np.interp(now, trace['angle_time'], trace[s + '_angle']) -
                       selected[s + '_angle_start_deg'] for s in ('pelvis', 'torso')]
            title.set_text(f'Swing {selected_id} | {ms:+.0f} ms')
            angle_text.set_text(f'Pelvis change: {changes[0]:+.1f} deg\nTorso change: {changes[1]:+.1f} deg')
            return lines + [trunk_line, cursor, title, angle_text]
        fig.tight_layout(rect=(0, 0, 1, .95))
        anim = FuncAnimation(fig, update_frame, frames=len(frame_times), interval=1000 / 60, blit=False)
        animation_html = anim.to_jshtml(fps=60)
        (FIG / 'selected_swing_animation.html').write_text(animation_html, encoding='utf-8')
        plt.close(fig)
        display(HTML(animation_html))

## Exploratory outcome relationships

Between-hitter Spearman correlations use one mean per hitter (at least three outcome-complete swings). Within-hitter associations subtract athlete/session means and correlate paired residuals, using groups with at least three paired swings. This removes group means but does not control pitch location or other confounders. Within-hitter estimates are pooled and swing-weighted; between-hitter estimates weight hitters equally.

Confidence intervals resample whole hitters. Sample sizes and valid bootstrap counts accompany every estimate. These are exploratory associations; no causal claim or automatic mechanical recommendation follows from a correlation. Measured attack angle is a bat metric, not the hand-path proxy from study 10.

In [ ]:
def correlation_estimate(data, x, y, mode):
    if mode == 'between_hitter':
        valid = data[['user', x, y]].dropna()
        grouped = valid.groupby('user')
        means = grouped[[x, y]].mean()
        means = means.loc[grouped.size().ge(3)].reset_index()
        units = [(np.array([r[x]]), np.array([r[y]])) for r in means.to_dict('records')]
    else:
        valid = data[['user', 'session', x, y]].dropna().copy()
        sizes = valid.groupby(['user', 'session'])[x].transform('size')
        valid = valid.loc[sizes.ge(3)].copy()
        valid[[x, y]] -= valid.groupby(['user', 'session'])[[x, y]].transform('mean')
        units = [(g[x].to_numpy(), g[y].to_numpy()) for _, g in valid.groupby('user')]
    n = sum(len(a) for a, _ in units)
    def rho(a, b):
        if len(a) < 3 or np.ptp(a) <= 1e-10 or np.ptp(b) <= 1e-10:
            return np.nan
        return float(spearmanr(a, b).statistic)
    if len(units) < 5:
        return dict(mode=mode, feature=x, outcome=y, hitters=len(units), observations=n,
                    rho=np.nan, ci_low=np.nan, ci_high=np.nan, valid_bootstraps=0)
    a = np.concatenate([u[0] for u in units]); b = np.concatenate([u[1] for u in units])
    estimate = rho(a, b)
    rng = np.random.default_rng(SEED)
    draws = []
    for _ in range(BOOTSTRAPS):
        chosen = rng.integers(0, len(units), len(units))
        draw = rho(np.concatenate([units[i][0] for i in chosen]),
                   np.concatenate([units[i][1] for i in chosen]))
        if np.isfinite(draw):
            draws.append(draw)
    low, high = np.quantile(draws, [.025, .975]) if len(draws) >= BOOTSTRAPS * .8 else (np.nan, np.nan)
    return dict(mode=mode, feature=x, outcome=y, hitters=len(units), observations=n,
                rho=estimate, ci_low=low, ci_high=high, valid_bootstraps=len(draws))

RELATIONSHIP_FEATURES = ['pelvis_pre_uprighting_deg', 'torso_pre_uprighting_deg',
                         'pelvis_bend_minus_rotation_ms', 'torso_bend_minus_rotation_ms']
relationships = pd.DataFrame([correlation_estimate(results, x, y, mode)
    for mode in ('between_hitter', 'within_hitter')
    for x in RELATIONSHIP_FEATURES for y in OUTCOMES])
display(relationships.round(3))

## Window, filter, and onset-threshold sensitivity

Window comparisons use the intersection of swings retained by all three windows. Signed changes across different windows naturally differ; their rank correlation and paired differences show that dependence. Filter and threshold comparisons use paired retained swings too. Onset comparisons include only swings with detected, uncensored onsets under both settings and report the paired count. They do not silently treat censored onsets as zero.

In [ ]:
window_results, window_audits = {}, []
for lookback in LOOKBACKS_S:
    table, review = (results.copy(), audit.copy()) if lookback == LOOKBACKS_S[0] else extract_cohort(lookback, CUTOFF_HZ)
    window_results[lookback] = table
    window_audits.append(review)
common_ids = set.intersection(*(set(t.session_swing) if len(t) else set() for t in window_results.values()))
sensitivity_rows = []
def compare_features(reference, alternate, label, features):
    if reference.empty or alternate.empty:
        return [dict(setting=label, feature=x, paired_swings=0, rank_rho=np.nan,
                     median_paired_difference=np.nan) for x in features]
    paired = reference[['session_swing'] + features].merge(
        alternate[['session_swing'] + features], on='session_swing', suffixes=('_base', '_alt'),
        validate='one_to_one')
    rows = []
    for feature in features:
        p = paired[[feature + '_base', feature + '_alt']].dropna()
        rank = spearmanr(p.iloc[:, 0], p.iloc[:, 1]).statistic if (
            len(p) >= 3 and p.iloc[:, 0].nunique() > 1 and p.iloc[:, 1].nunique() > 1) else np.nan
        rows.append(dict(setting=label, feature=feature, paired_swings=len(p), rank_rho=rank,
                         median_paired_difference=(p.iloc[:, 1] - p.iloc[:, 0]).median()))
    return rows

for lookback, table in window_results.items():
    sensitivity_rows += compare_features(results.loc[results.session_swing.isin(common_ids)],
        table.loc[table.session_swing.isin(common_ids)] if len(table) else table,
        f'lookback_{lookback * 1000:.0f}ms', RELATIONSHIP_FEATURES)
for cutoff in (8., 12., 16.):
    table, review = extract_cohort(LOOKBACKS_S[0], cutoff)
    review['setting'] = f'filter_{cutoff:g}Hz'; window_audits.append(review)
    sensitivity_rows += compare_features(results, table, f'filter_{cutoff:g}Hz',
                                         RELATIONSHIP_FEATURES + ['pelvis_pre_peak_rate_dps', 'torso_pre_peak_rate_dps'])
for upright_floor, rotation_floor in [(10., 40.), (30., 80.)]:
    table, review = extract_cohort(LOOKBACKS_S[0], CUTOFF_HZ, upright_floor, rotation_floor)
    label = f'onset_floors_{upright_floor:g}_{rotation_floor:g}dps'
    review['setting'] = label; window_audits.append(review)
    sensitivity_rows += compare_features(results, table, label,
        ['pelvis_bend_minus_rotation_ms', 'torso_bend_minus_rotation_ms'])
sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity.round(3))
print(f'Common window cohort: {len(common_ids)} swings.')

## Save outputs and interpret the run

The run manifest records settings, package versions, input hashes, and counts. CSVs include per-swing measurements, hitter/session profiles, onset status counts, outcome associations, sensitivity comparisons, and exclusion audits. An HTML animation and PNG figures accompany them.

Useful questions for a hitter are whether the change is abrupt, whether it precedes rotation, and whether it repeats across swings. A weak relationship with outcomes remains an informative result. A positive association does not demonstrate that changing posture will improve performance. The fixed window and threshold onsets are practical analytical definitions, not validated stride events.

In [ ]:
results.to_csv(OUT / 'swing_features.csv', index=False)
profiles.to_csv(OUT / 'hitter_session_profiles.csv', index=False)
audit.to_csv(OUT / 'primary_exclusion_audit.csv', index=False)
pd.concat(window_audits, ignore_index=True).to_csv(OUT / 'sensitivity_exclusion_audits.csv', index=False)
status_counts.to_csv(OUT / 'onset_status_counts.csv')
relationships.to_csv(OUT / 'outcome_relationships.csv', index=False)
sensitivity.to_csv(OUT / 'sensitivity_comparisons.csv', index=False)
for lookback, table in window_results.items():
    table.to_csv(OUT / f'features_{lookback * 1000:.0f}ms.csv', index=False)
input_names = ['joint_angles.csv', 'joint_velos.csv', 'metadata.csv', 'poi_metrics.csv']
if MAKE_SKELETON_ANIMATION:
    input_names.append('landmarks.csv')
manifest = dict(dataset='OpenBiomechanics Dataset v1', study='11_stride_bend_loss',
    input_hashes={n: sha256(RAW / n) for n in input_names},
    settings=dict(lookbacks_s=LOOKBACKS_S, cutoff_hz=CUTOFF_HZ,
                  upright_floor_dps=UPRIGHT_FLOOR_DPS, rotation_floor_dps=ROTATION_FLOOR_DPS,
                  onset_fraction=ONSET_FRACTION, sustain_s=SUSTAIN_S, bootstrap_draws=BOOTSTRAPS, seed=SEED),
    counts=dict(input_swings=len(swings), retained_swings=len(results), hitters=int(results.user.nunique()),
                common_window_swings=len(common_ids)), selected_swing=selected_id,
    packages={n: importlib.metadata.version(n) for n in ['numpy', 'pandas', 'scipy', 'matplotlib']},
    python=platform.python_version())
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
interpretation = [f'Retained **{len(results)} swings from {results.user.nunique()} hitters**.']
for segment in ('pelvis', 'torso'):
    change = results[segment + '_pre_uprighting_deg'].median()
    timing = results[segment + '_bend_minus_rotation_ms'].dropna()
    interpretation.append(f'{segment.title()}: median signed pre-contact change **{change:+.1f} degrees**; '
                          f'**{len(timing)}** swings have paired, uncensored onset estimates.')
display(Markdown('\n\n'.join(interpretation)))
print(f'CSV summaries: {OUT}')
print(f'Figures and animation: {FIG}')